[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 02](README.md)

# Pthreads: ciclo de vida y partición

**Tema:** 02 · **Sesiones:** 7 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo crear trabajo concurrente sin perder argumentos, errores ni cobertura de datos?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Crear hilos es sencillo; entregarles argumentos válidos, cubrir todo el dominio y reunir sus resultados exige un contrato preciso.

**Prerrequisitos.**

- Partición de datos y referencia serial del tema 01.
- Punteros, funciones y vida útil de objetos en C/C++.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Explicar create/join y la vida útil de argumentos.
- Particionar datos con cobertura comprobable.
- Comparar la salida paralela con una referencia serial.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

`pthread_create` inicia una función con un argumento cuya vida útil debe abarcar el acceso del hilo.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

`pthread_join` establece finalización y permite recuperar estado; ignorar códigos de retorno oculta fallos.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

La partición debe especificar rangos semiabiertos y funcionar cuando n no es múltiplo del número de hilos.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- carrera — accesos concurrentes incompatibles sin orden suficiente
- happens-before — relación que hace visible un efecto entre hilos
- deadlock — ciclo de espera que impide el progreso


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Fork Join

![Región serial que crea y reúne trabajadores](../../images/fork-join.svg)

**Cómo leerlo.** La región posterior al join solo puede consumir el resultado cuando todos los trabajadores necesarios terminaron y publicaron sus parciales.

### Distribucion Trabajo

![Iteraciones distribuidas y reducción final](../../images/distribucion-trabajo.svg)

**Cómo leerlo.** Verifica dos propiedades: cada iteración pertenece a un trabajador y la combinación de parciales reproduce la referencia serial.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "02"
NOTEBOOK = "02_memoria_compartida/01_pthreads.ipynb"
assert (ROOT / "curso" / "notebooks" / "02_memoria_compartida" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Rangos de trabajo

**Situación.** Se prueba una partición por bloques para casos irregulares.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
def partition(n, workers):
    q, r = divmod(n, workers)
    starts = [worker * q + min(worker, r) for worker in range(workers)]
    return [(start, start + q + (worker < r)) for worker, start in enumerate(starts)]
for n, workers in ((3, 5), (17, 4), (32, 8)):
    chunks = partition(n, workers)
    flattened = [i for begin, end in chunks for i in range(begin, end)]
    assert flattened == list(range(n))
    print(n, workers, chunks)


### Explicación del resultado

Los hilos sin elementos reciben un rango vacío válido; el programa no debe leer fuera de límites.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Referencia y reducción

**Situación.** Se simula la suma de parciales y se compara con una referencia única.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
values = [((i * 17) % 23) - 11 for i in range(101)]
chunks = partition(len(values), 6)
partials = [sum(values[begin:end]) for begin, end in chunks]
parallel_result = sum(partials)
reference = sum(values)
assert parallel_result == reference
print({"partials": partials, "result": parallel_result})


### Lectura razonada

En C, cada parcial debe tener almacenamiento independiente y la combinación ocurre después de join.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué ocurre con la cobertura cuando hay más trabajadores que elementos y cómo debe representarse ese caso?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Compilar y revisar `pthreads/thread_creation.c`.
2. Agregar comprobación de cada retorno de la API.
3. Probar n<threads, n no divisible y n grande.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Pasar la dirección de una variable de bucle compartida.
- Salir de `main` antes de join.
- Medir una versión paralela incorrecta.


## Criterios de aceptación

- Todos los retornos se comprueban.
- Cobertura de índices demostrada.
- Resultado igual a la referencia serial.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo crear trabajo concurrente sin perder argumentos, errores ni cobertura de datos?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Creación de hilos](../../../pthreads/thread_creation.c)
- [Ejemplo de mutex](../../../pthreads/thread_mutex.c)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 02](README.md)
